In [1]:
!pip install -q sentence-transformers

In [2]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from google.colab import files

In [4]:
uploaded = files.upload()

Saving policy_sbert_pipeline.zip to policy_sbert_pipeline.zip
Saving sbert_config.json to sbert_config.json


In [7]:
uploaded = files.upload()

Saving cleaned_policies.csv to cleaned_policies (1).csv


In [5]:
import zipfile
import json
import os
import pandas as pd
from sentence_transformers import SentenceTransformer

with open("sbert_config.json", "r") as f:
    config = json.load(f)

with zipfile.ZipFile("policy_sbert_pipeline.zip", "r") as zip_ref:
    zip_ref.extractall(".")

model = SentenceTransformer(config["model_path"])

print("Model:", config["model_name"])
print("Embedding dimension:", model.get_embedding_dimension())
print("Normalization:", config["normalize_embeddings"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Normalization: True


In [8]:
csv_file = next(file for file in uploaded if file.endswith(".csv"))
df = pd.read_csv(csv_file)

print("Policies:", len(df))
print(df.columns.tolist())

Policies: 4745
['Policy_ID', 'Policy_Name', 'Policy_Type', 'Department', 'Target_Group', 'Benefits_provided', 'Policy_Description', 'policy_text']


In [9]:
required_columns = ["Policy_ID", "policy_text"]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if df["Policy_ID"].isna().any():
    raise ValueError("Policy_ID contains missing values")

if df["Policy_ID"].duplicated().any():
    raise ValueError("Policy_ID contains duplicates")

if df["policy_text"].isna().any():
    raise ValueError("policy_text contains missing values")

if (df["policy_text"].str.strip() == "").any():
    raise ValueError("policy_text contains empty entries")

In [10]:
policy_ids = df["Policy_ID"].tolist()
texts = df["policy_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=config["normalize_embeddings"]
)

print("Generated embeddings:", embeddings.shape)

Batches:   0%|          | 0/149 [00:00<?, ?it/s]

Generated embeddings: (4745, 384)


In [11]:
embedding_df = pd.DataFrame({
    "Policy_ID": policy_ids,
    "embedding": [embedding.tolist() for embedding in embeddings]
})

embedding_df.to_csv("policy_embeddings.csv", index=False)

In [12]:
print(embedding_df.head())
print("Rows:", len(embedding_df))
print("Embedding size:", len(embedding_df.iloc[0]["embedding"]))

  Policy_ID                                          embedding
0        P1  [-0.008580905385315418, -0.036121565848588943,...
1        P2  [0.03674968332052231, -0.028446896001696587, -...
2        P3  [-0.02668333426117897, -0.02531810663640499, -...
3        P4  [-0.08132574707269669, 0.04407492280006409, -0...
4        P5  [-0.07241568714380264, 0.04909985139966011, -0...
Rows: 4745
Embedding size: 384


In [13]:
files.download("policy_embeddings.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>